# Environment

In [ ]:
import importlib.util
import subprocess
import sys

CLUSTER_RUN_DIR = '/path/to/ts-ifa/outputs/results/run_name'
COLAB_RUN_DIR = '/content/drive/MyDrive/path/to/outputs/results/run_name'
COLAB_REPO_DIR = '/content/drive/MyDrive/path/to/ts-ifa'
INSTALL_PROJECT_ON_COLAB = True
INSTALL_WIDGETS_IF_MISSING = True

try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if INSTALL_PROJECT_ON_COLAB:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q',
            '--no-deps', '-e', COLAB_REPO_DIR,
        ])
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', 'ipywidgets>=8.1',
        ])
elif INSTALL_WIDGETS_IF_MISSING and importlib.util.find_spec('ipywidgets') is None:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', 'ipywidgets>=8.1',
    ])

RUN_DIR = COLAB_RUN_DIR if IN_COLAB else CLUSTER_RUN_DIR
RUN_DIR

# Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import clear_output, display

from ts_ifa.visu.dashboard import (
    available_splits,
    gate_options,
    load_dashboard_data,
    plot_gate_roc,
    plot_horizon,
    plot_query_example,
    prediction_names,
    split_arrays,
)

# Data loading

In [ ]:
data = load_dashboard_data(RUN_DIR)
splits = available_splits(data)
print('Loaded splits:', ', '.join(splits))
print('Baseline visualization payload:', data['paths']['baseline'], data['paths']['baseline'].exists())
print('TS-IFA prediction payload:', data['paths']['ts_ifa'], data['paths']['ts_ifa'].exists())
for split in splits:
    arrays = split_arrays(data, split)
    print(f"{split}: {len(arrays['x']):,} queries; predictions={list(arrays['predictions'])}")

# Query and retrieved examples

In [ ]:
query_split = widgets.Dropdown(options=splits, value='eval' if 'eval' in splits else splits[0], description='split:')
query_sample = widgets.Button(description='random query', icon='random', button_style='primary')
query_normalized = widgets.ToggleButton(value=False, description='instance normalized', icon='exchange')
query_hide_axes = widgets.ToggleButton(value=False, description='remove axes', icon='eye-slash')
query_output = widgets.Output()
query_rng = np.random.default_rng()
query_state = {'index': 0}

def draw_query(*_):
    with query_output:
        clear_output(wait=True)
        fig = plot_query_example(
            data, query_split.value, query_state['index'],
            instance_normalized=query_normalized.value,
            hide_axes=query_hide_axes.value,
        )
        display(fig)
        plt.close(fig)

def sample_query(*_):
    count = len(split_arrays(data, query_split.value)['x'])
    query_state['index'] = int(query_rng.integers(count))
    draw_query()

query_sample.on_click(sample_query)
query_split.observe(sample_query, names='value')
query_normalized.observe(draw_query, names='value')
query_hide_axes.observe(draw_query, names='value')
display(widgets.VBox([
    widgets.HBox([query_split, query_sample]),
    widgets.HBox([query_normalized, query_hide_axes]),
    query_output,
]))
sample_query()

# Horizon-wise diagnostics

In [ ]:
horizon_split = widgets.Dropdown(options=splits, value='eval' if 'eval' in splits else splits[0], description='split:')
initial_names = prediction_names(data, horizon_split.value)
horizon_prediction = widgets.Dropdown(options=initial_names, value='vanilla' if 'vanilla' in initial_names else initial_names[0], description='quantity:')
horizon_reference = widgets.Dropdown(options=initial_names, value='vanilla' if 'vanilla' in initial_names else initial_names[0], description='relative to:')
horizon_metric = widgets.Dropdown(
    options=['raw prediction', 'direct difference', 'mse', 'nmse', 'relative mse'],
    value='mse', description='metric:',
)
horizon_normalized = widgets.ToggleButton(value=False, description='instance normalized', icon='exchange')
horizon_hide_axes = widgets.ToggleButton(value=False, description='remove axes', icon='eye-slash')
horizon_output = widgets.Output()

def draw_horizon(*_):
    with horizon_output:
        clear_output(wait=True)
        fig = plot_horizon(
            data, horizon_split.value, horizon_prediction.value, horizon_reference.value,
            horizon_metric.value, instance_normalized=horizon_normalized.value,
            hide_axes=horizon_hide_axes.value,
        )
        display(fig)
        plt.close(fig)

def update_horizon_names(*_):
    names = prediction_names(data, horizon_split.value)
    horizon_prediction.options = names
    horizon_reference.options = names
    horizon_prediction.value = 'vanilla' if 'vanilla' in names else names[0]
    horizon_reference.value = 'vanilla' if 'vanilla' in names else names[0]
    draw_horizon()

horizon_split.observe(update_horizon_names, names='value')
for control in [horizon_prediction, horizon_reference, horizon_metric, horizon_normalized, horizon_hide_axes]:
    control.observe(draw_horizon, names='value')
display(widgets.VBox([
    widgets.HBox([horizon_split, horizon_prediction]),
    widgets.HBox([horizon_reference, horizon_metric]),
    widgets.HBox([horizon_normalized, horizon_hide_axes]),
    horizon_output,
]))
draw_horizon()

# Gate ROC curves

In [ ]:
roc_splits = [split for split in splits if gate_options(data, split)]
roc_split = widgets.Dropdown(options=roc_splits or [''], value=(roc_splits or [''])[0], description='split:')
initial_gate_options = gate_options(data, roc_split.value) if roc_split.value else [('no gate payload', '')]
roc_gate = widgets.Dropdown(options=initial_gate_options, description='gate:')
roc_summary = widgets.HTML()
roc_output = widgets.Output()

def draw_roc(*_):
    with roc_output:
        clear_output(wait=True)
        if not roc_split.value or not roc_gate.value:
            roc_summary.value = '<b>No saved gate diagnostics.</b> Run evaluate_baselines first.'
            return
        fig, metrics = plot_gate_roc(data, roc_split.value, roc_gate.value)
        auc_text = f"{metrics['auc']:.4f}" if np.isfinite(metrics['auc']) else 'undefined (one class)'
        roc_summary.value = (
            f"<b>Accuracy:</b> {metrics['accuracy']:.4f} &nbsp; "
            f"<b>ROC AUC:</b> {auc_text} &nbsp; "
            f"<b>Decisions:</b> {int(metrics['count']):,}"
        )
        display(fig)
        plt.close(fig)

def update_gate_options(*_):
    roc_gate.options = gate_options(data, roc_split.value) if roc_split.value else [('no gate payload', '')]
    draw_roc()

roc_split.observe(update_gate_options, names='value')
roc_gate.observe(draw_roc, names='value')
display(widgets.VBox([
    widgets.HBox([roc_split, roc_gate]),
    roc_summary,
    roc_output,
]))
draw_roc()